# Post-Processing Ablation Study — OF_full_run_v2

This notebook evaluates the impact of **7 post-processing methods** on the
Onsets & Frames baseline model (OF_full_run_v2, 1000 epochs, full MAESTRO).

**Methods tested (incrementally):**
1. Onset-Conditioned Offset Estimation
2. Frame-Level Smoothing
3. Minimum Note Duration Enforcement
4. Velocity-Aware Duplicate Removal
5. Chord-Aware Onset Grouping
6. Adaptive Thresholding
7. Sustain Pedal-Aware Offset Extension

**Strategy:** Each experiment adds methods incrementally so we can see
the contribution of each. The final cell compares all configs.

**Important:** This notebook does NOT modify any original code.
It uses `decode_advanced.py` and `evaluate_advanced.py` alongside the originals.

## Cell 1 — Environment Setup

Run every session. Mounts Drive, installs packages, clones repo, sets paths.

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU! Change runtime to GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
# ── Clone / update repo ────────────────────────────────────────────────────
import os
from getpass import getpass

REPO_DIR = '/content/AMT_FYP'
TOKEN    = getpass('GitHub token: ')

if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://{TOKEN}@github.com/Mobinmo83/AMT_FYP.git {REPO_DIR}")
    print(f"Cloned → {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print(f"Pulled latest → {REPO_DIR}")

In [ ]:
%cd /content/AMT_FYP/piano_amt
!pip install -q -r requirements.txt
print("Packages installed ✓")

In [ ]:
# ── sys.path + project root ────────────────────────────────────────────────
import sys

PROJECT_ROOT = '/content/AMT_FYP/piano_amt'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Quick sanity check
from src.constants import N_MELS, FRAMES_PER_SECOND, N_KEYS
assert N_MELS == 229 and abs(FRAMES_PER_SECOND - 31.25) < 1e-6 and N_KEYS == 88

# Verify advanced modules exist
from models.onsets_frames.decode_advanced import advanced_rolls_to_note_events
from models.onsets_frames.evaluate_advanced import run_advanced_evaluation

print("All imports OK ✓")
print("  decode_advanced.py loaded ✓")
print("  evaluate_advanced.py loaded ✓")

## Cell 2 — Paths & Configuration

In [ ]:
# ── Path constants ──────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/piano_amt'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'

# ── Checkpoint ───────────────────────────────────────────────────────────────
BEST_CKPT = f'{RUNS_DIR}/OF_full_run_v2/checkpoints/best.pt'

# ── Evaluation settings ──────────────────────────────────────────────────────
SPLIT            = 'test'    # 'test' for final results, 'validation' for tuning
MODEL_COMPLEXITY = 48
MAX_FILES        = None      # None = all files in split

# ── Common eval protocol (locked — same for all experiments) ────────────────
EVAL_KWARGS = dict(
    checkpoint_path  = BEST_CKPT,
    maestro_root     = MAESTRO_ROOT,
    cache_dir        = CACHE_DIR,
    split            = SPLIT,
    max_files        = MAX_FILES,
    model_complexity = MODEL_COMPLEXITY,
    save_midi        = True,
    save_plots       = True,
    onset_threshold  = 0.5,
    frame_threshold  = 0.5,
    onset_tolerance  = 0.05,
    offset_ratio     = 0.2,
    offset_min_tolerance = 0.05,
    velocity_tolerance   = 0.1,
)

# Verify
import os, glob
assert os.path.exists(BEST_CKPT), f"Checkpoint not found: {BEST_CKPT}"
npzs = glob.glob(f'{CACHE_DIR}/*.npz')
print(f"Checkpoint : {BEST_CKPT}")
print(f"Cache files: {len(npzs)}")
print(f"Split      : {SPLIT}")
print(f"Max files  : {MAX_FILES or 'ALL'}")
print("\nReady ✓")

---
## Experiment 0 — Baseline (No Post-Processing)

Run the advanced evaluator with ALL methods OFF.
This should produce identical results to your original evaluation,
confirming the advanced code is correct.

In [ ]:
from models.onsets_frames.evaluate_advanced import run_advanced_evaluation

summary_baseline = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp0_baseline',
    # All methods OFF (defaults)
)

print("\n" + "=" * 60)
print("BASELINE DONE — verify these match your original eval_test results")
print("=" * 60)

---
## Experiment 1 — Method 3 Only: Minimum Note Duration

**Change:** Increase MIN_DUR from 16ms (original) to 50ms.

**Expected effect:** Removes very short spurious notes → fewer false positives → slight improvement in Note F1 and duplicate rate.

In [ ]:
summary_exp1 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp1_min_dur_50ms',
    # Method 3 ONLY
    min_note_duration_ms=50.0,
)

print("\nExperiment 1 DONE ✓")

---
## Experiment 2 — Methods 1 + 3: Onset-Conditioned Offset + Min Duration

**Added:** Method 1 — use the offset head to end notes earlier when offset fires.

**Expected effect:** Significant improvement in Note+Offset F1 and Offset MAE.

In [ ]:
summary_exp2 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp2_m1_m3',
    # Method 1
    use_onset_conditioned_offset=True,
    # Method 3
    min_note_duration_ms=50.0,
)

print("\nExperiment 2 DONE ✓")

---
## Experiment 3 — Methods 1 + 2 + 3: Add Frame Smoothing

**Added:** Method 2 — median filter on frame roll before decoding.

**Expected effect:** More stable note durations, fewer fragmented notes.

**Note:** Frame smoothing is applied BEFORE decoding, so it helps both
the standard frame-based offset and the onset-conditioned offset.

In [ ]:
summary_exp3 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp3_m1_m2_m3',
    # Method 1
    use_onset_conditioned_offset=True,
    # Method 2
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    # Method 3
    min_note_duration_ms=50.0,
)

print("\nExperiment 3 DONE ✓")

---
## Experiment 4 — Methods 1+2+3+4+5: Add Duplicate Removal + Chord Grouping

**Added:** 
- Method 4 — remove duplicate onsets on same pitch within 50ms
- Method 5 — snap near-simultaneous onsets to median time (30ms window)

**Expected effect:** Cleaner chord structure, fewer duplicates.

In [ ]:
summary_exp4 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp4_m1_to_m5',
    # Method 1
    use_onset_conditioned_offset=True,
    # Method 2
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    # Method 3
    min_note_duration_ms=50.0,
    # Method 4
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
    # Method 5
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
)

print("\nExperiment 4 DONE ✓")

---
## Experiment 5 — Methods 1–6: Add Adaptive Thresholding

**Added:** Method 6 — per-piece threshold adaptation based on activation statistics.

**Expected effect:** Better handling of pieces with varying dynamics.
May help or hurt — this is why we test incrementally.

In [ ]:
summary_exp5 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp5_m1_to_m6',
    # Method 1
    use_onset_conditioned_offset=True,
    # Method 2
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    # Method 3
    min_note_duration_ms=50.0,
    # Method 4
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
    # Method 5
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
    # Method 6
    use_adaptive_thresholds=True,
    adaptive_onset_k=0.5,
    adaptive_frame_k=0.5,
)

print("\nExperiment 5 DONE ✓")

---
## Experiment 6 — All Methods (1–7): Add Sustain Pedal Extension

**Added:** Method 7 — extend note offsets in likely-pedaled regions.

**Expected effect:** Uncertain — may improve offset accuracy in sustained passages
but could also over-extend notes. This is the experimental one.

In [ ]:
summary_exp6 = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp6_all_methods',
    # Method 1
    use_onset_conditioned_offset=True,
    # Method 2
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    # Method 3
    min_note_duration_ms=50.0,
    # Method 4
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
    # Method 5
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
    # Method 6
    use_adaptive_thresholds=True,
    adaptive_onset_k=0.5,
    adaptive_frame_k=0.5,
    # Method 7
    use_pedal_extension=True,
    pedal_energy_threshold=10.0,
    pedal_max_extension_sec=2.0,
)

print("\nExperiment 6 DONE ✓")

---
## Final Comparison — All Experiments

Collect all results and produce a comparison table + chart.
This uses your existing `compare.py` which scans `summary_metrics.json` files.

In [ ]:
import json
import pandas as pd
from pathlib import Path

# ── Collect all experiment summaries ────────────────────────────────────────
run_dir = Path(RUNS_DIR) / 'OF_full_run_v2'

# Find all eval directories (both original and advanced)
eval_dirs = sorted(run_dir.glob(f'eval_{SPLIT}*'))

print(f"Found {len(eval_dirs)} evaluation configs:\n")
for d in eval_dirs:
    print(f"  {d.name}")

# ── Load summaries ──────────────────────────────────────────────────────────
rows = []
for d in eval_dirs:
    summary_file = d / 'summary_metrics.json'
    if summary_file.exists():
        with open(summary_file) as f:
            data = json.load(f)
        data['config'] = d.name.replace(f'eval_{SPLIT}_', '').replace(f'eval_{SPLIT}', 'original')
        rows.append(data)

print(f"\nLoaded {len(rows)} summaries")

In [ ]:
# ── Build comparison table ──────────────────────────────────────────────────

# Primary metrics comparison
print("\n" + "=" * 100)
print("POST-PROCESSING ABLATION RESULTS — PRIMARY METRICS")
print("=" * 100)
print(f"\n{'Config':<25s}  {'Note F1':>8s}  {'N+Off F1':>9s}  {'N+O+V F1':>9s}  {'Frame F1':>9s}  {'Adv Note':>9s}  {'Adv N+O':>9s}  {'Adv N+O+V':>10s}")
print("-" * 100)

for row in rows:
    config = row.get('config', '?')
    # Original metrics (from compute_metrics using original decode)
    note_f1     = row.get('note_f1', 0)
    noff_f1     = row.get('note_with_offset_f1', 0)
    noffv_f1    = row.get('note_with_offset_vel_f1', 0)
    frame_f1    = row.get('frame_f1', 0)
    # Advanced metrics (from advanced decode)
    adv_note    = row.get('adv_note_f1', note_f1)
    adv_noff    = row.get('adv_note_with_offset_f1', noff_f1)
    adv_noffv   = row.get('adv_note_with_offset_vel_f1', noffv_f1)
    
    print(f"{config:<25s}  {note_f1:>8.4f}  {noff_f1:>9.4f}  {noffv_f1:>9.4f}  {frame_f1:>9.4f}  {adv_note:>9.4f}  {adv_noff:>9.4f}  {adv_noffv:>10.4f}")

print("\n")

# Supplementary metrics comparison
print("=" * 80)
print("POST-PROCESSING ABLATION RESULTS — SUPPLEMENTARY METRICS")
print("=" * 80)
print(f"\n{'Config':<25s}  {'Off MAE(ms)':>11s}  {'On MAE(ms)':>10s}  {'Chord Comp':>10s}  {'Dup Rate':>8s}")
print("-" * 80)

for row in rows:
    config   = row.get('config', '?')
    off_mae  = row.get('ea_offset_mae_ms', 0)
    on_mae   = row.get('ea_onset_mae_ms', 0)
    chord_c  = row.get('ea_chord_completeness', 0)
    dup_rate = row.get('ea_duplicate_note_rate', 0)
    
    print(f"{config:<25s}  {off_mae:>11.1f}  {on_mae:>10.1f}  {chord_c:>10.4f}  {dup_rate:>8.4f}")

print()

In [ ]:
# ── Improvement Delta Table ─────────────────────────────────────────────────
# Show how much each experiment improved over baseline

baseline = None
for row in rows:
    if 'baseline' in row.get('config', '') or row.get('config', '') == 'original':
        baseline = row
        break

if baseline is None and rows:
    baseline = rows[0]  # Use first as baseline

if baseline:
    print("\n" + "=" * 90)
    print("IMPROVEMENT OVER BASELINE (Δ values)")
    print("=" * 90)
    
    base_note   = baseline.get('adv_note_f1', baseline.get('note_f1', 0))
    base_noff   = baseline.get('adv_note_with_offset_f1', baseline.get('note_with_offset_f1', 0))
    base_noffv  = baseline.get('adv_note_with_offset_vel_f1', baseline.get('note_with_offset_vel_f1', 0))
    
    print(f"\nBaseline: {baseline.get('config', '?')}")
    print(f"  Note F1={base_note:.4f}  N+Off F1={base_noff:.4f}  N+O+V F1={base_noffv:.4f}")
    print(f"\n{'Config':<25s}  {'Δ Note F1':>10s}  {'Δ N+Off F1':>11s}  {'Δ N+O+V F1':>11s}")
    print("-" * 65)
    
    for row in rows:
        config = row.get('config', '?')
        adv_note  = row.get('adv_note_f1', row.get('note_f1', 0))
        adv_noff  = row.get('adv_note_with_offset_f1', row.get('note_with_offset_f1', 0))
        adv_noffv = row.get('adv_note_with_offset_vel_f1', row.get('note_with_offset_vel_f1', 0))
        
        d_note  = adv_note - base_note
        d_noff  = adv_noff - base_noff
        d_noffv = adv_noffv - base_noffv
        
        sn = '+' if d_note >= 0 else ''
        so = '+' if d_noff >= 0 else ''
        sv = '+' if d_noffv >= 0 else ''
        
        print(f"{config:<25s}  {sn}{d_note:>9.4f}  {so}{d_noff:>10.4f}  {sv}{d_noffv:>10.4f}")
    
    print()

In [ ]:
# ── Visualisation: Bar chart comparing all configs ──────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image

configs = [row.get('config', '?') for row in rows]
metrics_to_plot = [
    ('adv_note_f1', 'note_f1', 'Note F1'),
    ('adv_note_with_offset_f1', 'note_with_offset_f1', 'Note+Off F1'),
    ('adv_note_with_offset_vel_f1', 'note_with_offset_vel_f1', 'N+O+V F1'),
]

x = np.arange(len(metrics_to_plot))
n_configs = len(configs)
width = 0.8 / max(n_configs, 1)
colors = plt.cm.tab10(np.linspace(0, 1, max(n_configs, 1)))

fig, ax = plt.subplots(figsize=(14, 6))

for i, (row, color) in enumerate(zip(rows, colors)):
    vals = []
    for adv_key, orig_key, _ in metrics_to_plot:
        vals.append(row.get(adv_key, row.get(orig_key, 0)))
    offset = (i - n_configs / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=row.get('config', '?'),
                  color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{v:.3f}", ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metrics_to_plot], fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title(f'Post-Processing Ablation — OF_full_run_v2 ({SPLIT} split)', fontsize=14)
ax.legend(fontsize=7, loc='lower right', ncol=2)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()

save_path = str(run_dir / f'postprocessing_ablation_{SPLIT}.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Chart saved → {save_path}")
plt.close(fig)

Image(save_path)

---
## Decision: Choose Best Post-Processing Configuration

Based on the results above, identify:
1. Which individual method helped the most?
2. Which combination gave the best overall results?
3. Did any method actually hurt performance?

Use the best config as your final post-processing pipeline
before adding the Transformer refiner.

In [ ]:
# ── Find best config by Note+Offset F1 (the main target metric) ────────────

best_config = None
best_noff = 0

for row in rows:
    noff = row.get('adv_note_with_offset_f1', row.get('note_with_offset_f1', 0))
    if noff > best_noff:
        best_noff = noff
        best_config = row

if best_config:
    print("=" * 60)
    print("BEST POST-PROCESSING CONFIGURATION")
    print("=" * 60)
    print(f"\n  Config: {best_config.get('config', '?')}")
    print(f"  Note+Offset F1: {best_noff:.4f}")
    
    # Show which methods were active
    pp = best_config.get('post_processing', {})
    if pp:
        print(f"\n  Active methods:")
        if pp.get('use_onset_conditioned_offset'):
            print(f"    ✓ Method 1: Onset-Conditioned Offset")
        if pp.get('use_frame_smoothing'):
            print(f"    ✓ Method 2: Frame Smoothing (kernel={pp.get('frame_smoothing_kernel')})")
        if pp.get('min_note_duration_ms', 16) != 16:
            print(f"    ✓ Method 3: Min Duration = {pp.get('min_note_duration_ms')}ms")
        if pp.get('use_duplicate_removal'):
            print(f"    ✓ Method 4: Duplicate Removal")
        if pp.get('use_chord_grouping'):
            print(f"    ✓ Method 5: Chord Grouping")
        if pp.get('use_adaptive_thresholds'):
            print(f"    ✓ Method 6: Adaptive Thresholds")
        if pp.get('use_pedal_extension'):
            print(f"    ✓ Method 7: Pedal Extension")
    
    print(f"\n  → Use this config as input to your Transformer refiner.")
    print("=" * 60)

In [ ]:
# ── Save final comparison to CSV for dissertation ────────────────────────────

comparison_rows = []
for row in rows:
    comparison_rows.append({
        'Config': row.get('config', '?'),
        'Note F1': row.get('adv_note_f1', row.get('note_f1', 0)),
        'Note+Off F1': row.get('adv_note_with_offset_f1', row.get('note_with_offset_f1', 0)),
        'N+O+V F1': row.get('adv_note_with_offset_vel_f1', row.get('note_with_offset_vel_f1', 0)),
        'Frame F1': row.get('frame_f1', 0),
        'Offset MAE (ms)': row.get('ea_offset_mae_ms', 0),
        'Chord Comp': row.get('ea_chord_completeness', 0),
        'Dup Rate': row.get('ea_duplicate_note_rate', 0),
        'Eval Time (s)': row.get('eval_time_total_s', 0),
    })

df_comparison = pd.DataFrame(comparison_rows)
csv_path = str(run_dir / f'postprocessing_ablation_{SPLIT}.csv')
df_comparison.to_csv(csv_path, index=False)
print(f"Comparison CSV saved → {csv_path}")
print()
print(df_comparison.to_string(index=False))